In [1]:
# ============================================================
# OPTUNA TUNING - CELL 1: Setup and Load Required Data
# ============================================================

import numpy as np
import pandas as pd
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from tensorflow.keras.models import load_model
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold

# Install optuna if not already available
try:
    import optuna
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'optuna', '--quiet'])
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)  # keeps output clean

PROJECT_ROOT = "C:/Users/USER/Desktop/Solar_Energy_Forecasting_Project"
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results")
MODELS_DIR   = os.path.join(PROJECT_ROOT, "models")
SEQ_DIR      = os.path.join(RESULTS_DIR, "sequences")

CITIES = ['kano', 'maid', 'abuja']
CITY_LABELS = {'kano': 'Kano', 'maid': 'Maiduguri', 'abuja': 'Abuja'}

data = {}
for city in CITIES:
    data[city] = {
        'X_train': np.load(os.path.join(SEQ_DIR, f'X_train_{city}.npy')),
        'y_train': np.load(os.path.join(SEQ_DIR, f'y_train_{city}.npy')),
        'X_val':   np.load(os.path.join(SEQ_DIR, f'X_val_{city}.npy')),
        'y_val':   np.load(os.path.join(SEQ_DIR, f'y_val_{city}.npy')),
        'X_test':  np.load(os.path.join(SEQ_DIR, f'X_test_{city}.npy')),
        'y_test':  np.load(os.path.join(SEQ_DIR, f'y_test_{city}.npy')),
        'tgt_scaler': joblib.load(os.path.join(MODELS_DIR, f'tgt_scaler_{city}.pkl')),
    }

print("✅ Data loaded for all 3 cities")

✅ Data loaded for all 3 cities


In [2]:
# ============================================================
# OPTUNA TUNING - CELL 2: Evaluation Function
# ============================================================

def evaluate_model(y_pred_norm, y_test_norm, tgt_scaler):
    y_pred_kw = tgt_scaler.inverse_transform(y_pred_norm.reshape(-1, 1)).flatten()
    y_true_kw = tgt_scaler.inverse_transform(y_test_norm.reshape(-1, 1)).flatten()

    mask = y_true_kw > 0
    y_true_day = y_true_kw[mask]
    y_pred_day = y_pred_kw[mask]

    rmse = np.sqrt(mean_squared_error(y_true_day, y_pred_day))
    mae  = mean_absolute_error(y_true_day, y_pred_day)
    mape = np.mean(np.abs((y_true_day - y_pred_day) / (y_true_day + 1e-8))) * 100
    r2   = r2_score(y_true_day, y_pred_day)
    mean_actual = y_true_day.mean()

    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2,
            'nRMSE': (rmse/mean_actual)*100, 'nMAE': (mae/mean_actual)*100}

results = {city: {} for city in CITIES}

In [3]:
# ============================================================
# OPTUNA TUNING - CELL 3: Rebuild v3 (34-feature) Feature Set
# ============================================================

FEATURE_NAMES_17 = [
    'GHI', 'DHI', 'Clearness', 'Temperature', 'Humidity', 'WindSpeed',
    'Hour_sin', 'Hour_cos', 'Month_sin', 'Month_cos',
    'DayOfYear_sin', 'DayOfYear_cos', 'Season_encoded', 'Is_Daytime',
    'PV_Power_Lag1', 'PV_Power_Lag24', 'GHI_Lag1'
]
FEATURE_INDEX = {name: i for i, name in enumerate(FEATURE_NAMES_17)}
GHI_IDX, HUMIDITY_IDX, CLEARNESS_IDX = FEATURE_INDEX['GHI'], FEATURE_INDEX['Humidity'], FEATURE_INDEX['Clearness']

def build_v3_features(X_sequences, cnn_pred_norm):
    last_timestep = X_sequences[:, -1, :]
    cnn_pred_col = cnn_pred_norm.reshape(-1, 1)

    rolling_features = []
    for window in [3, 6]:
        recent_window = X_sequences[:, -window:, :]
        for feat_idx in [GHI_IDX, HUMIDITY_IDX, CLEARNESS_IDX]:
            rolling_features.append(recent_window[:, :, feat_idx].mean(axis=1))
            rolling_features.append(recent_window[:, :, feat_idx].std(axis=1))
    rolling_matrix = np.column_stack(rolling_features)

    season, clearness = last_timestep[:, FEATURE_INDEX['Season_encoded']], last_timestep[:, CLEARNESS_IDX]
    month_cos, doy_cos = last_timestep[:, FEATURE_INDEX['Month_cos']], last_timestep[:, FEATURE_INDEX['DayOfYear_cos']]
    humidity, ghi = last_timestep[:, HUMIDITY_IDX], last_timestep[:, GHI_IDX]

    interactions = np.column_stack([
        season * clearness, doy_cos * humidity, month_cos * ghi, season * humidity
    ])

    return np.hstack([last_timestep, cnn_pred_col, rolling_matrix, interactions])

# Build features for all 3 cities
v3_data = {}
for city in CITIES:
    print(f"Building v3 features — {CITY_LABELS[city]}...")
    cnn_bilstm_model = load_model(os.path.join(MODELS_DIR, f'best_cnn_bilstm_{CITY_LABELS[city].lower()}.keras'))

    cnn_pred_train = cnn_bilstm_model.predict(data[city]['X_train'], verbose=0).flatten()
    cnn_pred_val   = cnn_bilstm_model.predict(data[city]['X_val'],   verbose=0).flatten()
    cnn_pred_test  = cnn_bilstm_model.predict(data[city]['X_test'],  verbose=0).flatten()

    v3_data[city] = {
        'X_train': build_v3_features(data[city]['X_train'], cnn_pred_train),
        'X_val':   build_v3_features(data[city]['X_val'],   cnn_pred_val),
        'X_test':  build_v3_features(data[city]['X_test'],  cnn_pred_test),
        'cnn_pred_train': cnn_pred_train, 'cnn_pred_val': cnn_pred_val, 'cnn_pred_test': cnn_pred_test,
    }
    print(f"  ✅ {v3_data[city]['X_train'].shape[1]} features built\n")

Building v3 features — Kano...
  ✅ 34 features built

Building v3 features — Maiduguri...
  ✅ 34 features built

Building v3 features — Abuja...
  ✅ 34 features built



In [4]:
# ============================================================
# OPTUNA TUNING - CELL 4: Hyperparameter Search (per city)
# ============================================================

N_TRIALS = 60   # how many different hyperparameter combinations to try per city

def objective(trial, city):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 200, 600),
        'max_depth'         : trial.suggest_int('max_depth', 3, 8),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample'         : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight'  : trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 0.001, 1.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 0.1, 3.0, log=True),
        'objective'         : 'reg:squarederror',
        'random_state'      : 42,
        'n_jobs'            : -1,
        'verbosity'         : 0
    }

    residual_train = data[city]['y_train'] - v3_data[city]['cnn_pred_train']
    residual_val   = data[city]['y_val']   - v3_data[city]['cnn_pred_val']

    model = xgb.XGBRegressor(**params)
    model.fit(v3_data[city]['X_train'], residual_train,
              eval_set=[(v3_data[city]['X_val'], residual_val)], verbose=False)

    residual_pred_val = model.predict(v3_data[city]['X_val'])
    hybrid_val_pred = v3_data[city]['cnn_pred_val'] + residual_pred_val

    # Score on daytime-only RMSE — same convention as everything else
    y_true_kw = data[city]['tgt_scaler'].inverse_transform(data[city]['y_val'].reshape(-1,1)).flatten()
    y_pred_kw = data[city]['tgt_scaler'].inverse_transform(hybrid_val_pred.reshape(-1,1)).flatten()
    mask = y_true_kw > 0
    rmse = np.sqrt(mean_squared_error(y_true_kw[mask], y_pred_kw[mask]))
    return rmse

best_params = {}
xgb_models_optuna = {}

for city in CITIES:
    print(f"\n{'='*55}\nOPTUNA SEARCH — {CITY_LABELS[city]} ({N_TRIALS} trials)\n{'='*55}")

    study = optuna.create_study(direction='minimize')
    study.optimize(lambda trial: objective(trial, city), n_trials=N_TRIALS, show_progress_bar=True)

    best_params[city] = study.best_params
    print(f"\nBest validation RMSE: {study.best_value:.3f}")
    print(f"Best params: {study.best_params}")

    # Retrain final model on train+val using best found settings
    final_params = {**study.best_params, 'objective': 'reg:squarederror',
                     'random_state': 42, 'n_jobs': -1, 'verbosity': 0}
    residual_train = data[city]['y_train'] - v3_data[city]['cnn_pred_train']
    residual_val   = data[city]['y_val']   - v3_data[city]['cnn_pred_val']

    final_model = xgb.XGBRegressor(**final_params)
    final_model.fit(v3_data[city]['X_train'], residual_train,
                     eval_set=[(v3_data[city]['X_val'], residual_val)], verbose=False)

    xgb_models_optuna[city] = final_model
    joblib.dump(final_model, os.path.join(MODELS_DIR, f'xgb_{city}_v3_optuna.pkl'))


OPTUNA SEARCH — Kano (60 trials)


  0%|          | 0/60 [00:00<?, ?it/s]


Best validation RMSE: 32.626
Best params: {'n_estimators': 391, 'max_depth': 5, 'learning_rate': 0.14174092444331768, 'subsample': 0.909059258047136, 'colsample_bytree': 0.6846507395181687, 'min_child_weight': 9, 'reg_alpha': 0.710168761239696, 'reg_lambda': 0.6316340654376424}

OPTUNA SEARCH — Maiduguri (60 trials)


  0%|          | 0/60 [00:00<?, ?it/s]


Best validation RMSE: 31.315
Best params: {'n_estimators': 599, 'max_depth': 6, 'learning_rate': 0.03463363673236754, 'subsample': 0.6290400852295179, 'colsample_bytree': 0.6209885716392106, 'min_child_weight': 3, 'reg_alpha': 0.7388816961179141, 'reg_lambda': 1.294296359401778}

OPTUNA SEARCH — Abuja (60 trials)


  0%|          | 0/60 [00:00<?, ?it/s]


Best validation RMSE: 32.768
Best params: {'n_estimators': 519, 'max_depth': 5, 'learning_rate': 0.06577966702967675, 'subsample': 0.6005457733969071, 'colsample_bytree': 0.6291115003330303, 'min_child_weight': 4, 'reg_alpha': 0.9889726307074751, 'reg_lambda': 0.29719006674430193}


In [5]:
# ============================================================
# OPTUNA TUNING - CELL 5: Evaluate Tuned Model on Test Set
# ============================================================

for city in CITIES:
    residual_pred_test = xgb_models_optuna[city].predict(v3_data[city]['X_test'])
    hybrid_v3_optuna_pred = v3_data[city]['cnn_pred_test'] + residual_pred_test

    results[city]['hybrid_v3_optuna'] = evaluate_model(
        hybrid_v3_optuna_pred, data[city]['y_test'], data[city]['tgt_scaler']
    )

    m = results[city]['hybrid_v3_optuna']
    print(f"{CITY_LABELS[city]} — Hybrid v3 (Optuna-tuned): "
          f"RMSE={m['RMSE']:.2f} | MAE={m['MAE']:.2f} | MAPE={m['MAPE']:.2f} | R2={m['R2']:.4f}")

Kano — Hybrid v3 (Optuna-tuned): RMSE=30.25 | MAE=16.89 | MAPE=9.14 | R2=0.9836
Maiduguri — Hybrid v3 (Optuna-tuned): RMSE=33.33 | MAE=18.80 | MAPE=13.87 | R2=0.9798
Abuja — Hybrid v3 (Optuna-tuned): RMSE=31.06 | MAE=18.99 | MAPE=11.76 | R2=0.9809


In [6]:
# ============================================================
# OPTUNA TUNING - CELL 6: Rebuild v2 (30-feature) Set
# ============================================================

def build_v2_features(X_sequences, cnn_pred_norm):
    last_timestep = X_sequences[:, -1, :]
    cnn_pred_col = cnn_pred_norm.reshape(-1, 1)

    rolling_features = []
    for window in [3, 6]:
        recent_window = X_sequences[:, -window:, :]
        for feat_idx in [GHI_IDX, HUMIDITY_IDX, CLEARNESS_IDX]:
            rolling_features.append(recent_window[:, :, feat_idx].mean(axis=1))
            rolling_features.append(recent_window[:, :, feat_idx].std(axis=1))
    rolling_matrix = np.column_stack(rolling_features)

    return np.hstack([last_timestep, cnn_pred_col, rolling_matrix])

v2_data = {}
for city in CITIES:
    v2_data[city] = {
        'X_train': build_v2_features(data[city]['X_train'], v3_data[city]['cnn_pred_train']),
        'X_test':  build_v2_features(data[city]['X_test'],  v3_data[city]['cnn_pred_test']),
    }
print("✅ v2 (30-feature) sets rebuilt")

✅ v2 (30-feature) sets rebuilt


In [7]:
# ============================================================
# OPTUNA TUNING - CELL 7: Evaluate All Other Saved Models
# ============================================================

for city in CITIES:
    # ── Standalone XGBoost ──
    xgb_only_path = os.path.join(MODELS_DIR, f'xgb_only_{city}.pkl')
    if os.path.exists(xgb_only_path):
        xgb_only_model = joblib.load(xgb_only_path)
        X_test_flat = data[city]['X_test'][:, -1, :]
        y_pred = xgb_only_model.predict(X_test_flat)
        results[city]['xgb_only'] = evaluate_model(y_pred, data[city]['y_test'], data[city]['tgt_scaler'])

    # ── Original Hybrid (Stage 5's saved predictions) ──
    hybrid_pred_path = os.path.join(RESULTS_DIR, f'y_pred_hybrid_{city}.npy')
    if os.path.exists(hybrid_pred_path):
        y_pred_hybrid_kw = np.load(hybrid_pred_path)
        y_true_kw = data[city]['tgt_scaler'].inverse_transform(data[city]['y_test'].reshape(-1,1)).flatten()
        mask = y_true_kw > 0
        rmse = np.sqrt(mean_squared_error(y_true_kw[mask], y_pred_hybrid_kw[mask]))
        mae  = mean_absolute_error(y_true_kw[mask], y_pred_hybrid_kw[mask])
        mape = np.mean(np.abs((y_true_kw[mask]-y_pred_hybrid_kw[mask])/(y_true_kw[mask]+1e-8)))*100
        r2   = r2_score(y_true_kw[mask], y_pred_hybrid_kw[mask])
        results[city]['hybrid'] = {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

    # ── Hybrid v2 ──
    v2_model_path = os.path.join(MODELS_DIR, f'xgb_{city}_v2.pkl')
    if os.path.exists(v2_model_path):
        v2_model = joblib.load(v2_model_path)
        residual_pred = v2_model.predict(v2_data[city]['X_test'])
        hybrid_v2_pred = v3_data[city]['cnn_pred_test'] + residual_pred
        results[city]['hybrid_v2'] = evaluate_model(hybrid_v2_pred, data[city]['y_test'], data[city]['tgt_scaler'])

    # ── Hybrid v3 (manual) ──
    v3_model_path = os.path.join(MODELS_DIR, f'xgb_{city}_v3.pkl')
    if os.path.exists(v3_model_path):
        v3_model = joblib.load(v3_model_path)
        residual_pred = v3_model.predict(v3_data[city]['X_test'])
        hybrid_v3_pred = v3_data[city]['cnn_pred_test'] + residual_pred
        results[city]['hybrid_v3'] = evaluate_model(hybrid_v3_pred, data[city]['y_test'], data[city]['tgt_scaler'])

print("✅ All saved models loaded and evaluated")

✅ All saved models loaded and evaluated


In [8]:
# ============================================================
# OPTUNA TUNING - CELL 8: Load Multi-Run Averaged Baselines
# ============================================================

multi_run_df = pd.read_csv(os.path.join(RESULTS_DIR, 'multi_run_validated_comparison.csv'))
print("✅ Multi-run averaged baselines loaded")
print(multi_run_df[['City', 'Model']].drop_duplicates().to_string(index=False))

✅ Multi-run averaged baselines loaded
     City                                  Model
     Kano                         Standalone CNN
     Kano                        Standalone LSTM
     Kano             CNN-LSTM (uni-directional)
     Kano                     Standalone XGBoost
     Kano Proposed Hybrid (CNN-biLSTM + XGBoost)
Maiduguri                         Standalone CNN
Maiduguri                        Standalone LSTM
Maiduguri             CNN-LSTM (uni-directional)
Maiduguri                     Standalone XGBoost
Maiduguri Proposed Hybrid (CNN-biLSTM + XGBoost)
    Abuja                         Standalone CNN
    Abuja                        Standalone LSTM
    Abuja             CNN-LSTM (uni-directional)
    Abuja                     Standalone XGBoost
    Abuja Proposed Hybrid (CNN-biLSTM + XGBoost)


In [12]:
# ============================================================
# OPTUNA TUNING - CELL 9: Full Final Comparison Table
# ============================================================

MODEL_LABELS = {
    'xgb_only':          'Standalone XGBoost',
    'hybrid':            'Original Hybrid',
    'hybrid_v2':         'Hybrid v2 (Enhanced Features)',
    'hybrid_v3':         'Hybrid v3 (Interactions + Tuned)',
    'hybrid_v3_optuna':  'Hybrid v3 (Optuna-Tuned)',
}

final_rows = []

for city in CITIES:
    # Optuna-computed models from this notebook
    for key in ['xgb_only', 'hybrid', 'hybrid_v2', 'hybrid_v3', 'hybrid_v3_optuna']:
        if key in results[city]:
            m = results[city][key]
            final_rows.append({
                'City': CITY_LABELS[city], 'Model': MODEL_LABELS[key],
                'RMSE (kW)': round(m['RMSE'], 2), 'MAE (kW)': round(m['MAE'], 2),
                'MAPE (%)': round(m['MAPE'], 2), 'R2': round(m['R2'], 4),
            })

    # Multi-run averaged neural network baselines
    city_multi = multi_run_df[
        (multi_run_df['City'] == CITY_LABELS[city]) &
        (multi_run_df['Model'].str.contains('CNN|LSTM', regex=True)) &
        (~multi_run_df['Model'].str.contains('Hybrid|XGBoost'))
    ]
    for _, row in city_multi.iterrows():
        final_rows.append({
            'City': CITY_LABELS[city], 'Model': f"{row['Model']} (5-run avg)",
            'RMSE (kW)': f"{row['RMSE_mean']:.2f} ± {row['RMSE_std']:.2f}",
            'MAE (kW)':  f"{row['MAE_mean']:.2f} ± {row['MAE_std']:.2f}",
            'MAPE (%)':  f"{row['MAPE_mean']:.2f} ± {row['MAPE_std']:.2f}",
            'R2':        f"{row['R2_mean']:.4f} ± {row['R2_std']:.4f}",
        })

final_df = pd.DataFrame(final_rows)
final_df.to_csv(os.path.join(RESULTS_DIR, 'final_complete_comparison.csv'), index=False)

for city in CITIES:
    print(f"\n{'='*80}\nFINAL COMPLETE COMPARISON — {CITY_LABELS[city]}\n{'='*80}")
    print(final_df[final_df['City']==CITY_LABELS[city]].drop(columns='City').to_string(index=False))

print(f"\n✅ Saved to: final_complete_comparison.csv")


FINAL COMPLETE COMPARISON — Kano
                                 Model    RMSE (kW)     MAE (kW)     MAPE (%)              R2
                    Standalone XGBoost        29.81        17.08         11.3          0.9841
                       Original Hybrid        30.55        17.98        11.59          0.9833
         Hybrid v2 (Enhanced Features)        30.52        17.92        11.66          0.9833
      Hybrid v3 (Interactions + Tuned)        30.19        17.31         10.3          0.9836
              Hybrid v3 (Optuna-Tuned)        30.25        16.89         9.14          0.9836
            Standalone CNN (5-run avg) 53.56 ± 9.29 40.13 ± 8.98 28.92 ± 6.47 0.9470 ± 0.0193
           Standalone LSTM (5-run avg) 32.75 ± 2.08 21.24 ± 2.41 11.52 ± 2.81 0.9807 ± 0.0024
CNN-LSTM (uni-directional) (5-run avg) 44.01 ± 4.69 31.99 ± 3.85 28.40 ± 2.53 0.9649 ± 0.0075

FINAL COMPLETE COMPARISON — Maiduguri
                                 Model    RMSE (kW)     MAE (kW)     MAPE (%)    

In [13]:
# ============================================================
# OPTUNA TUNING - CELL 10: Add Composite Ranking
# ============================================================

# Rebuild with RAW numeric values (not yet formatted as strings)
# so ranking works correctly on the actual numbers
ranking_rows = []

for city in CITIES:
    for key in ['xgb_only', 'hybrid', 'hybrid_v2', 'hybrid_v3', 'hybrid_v3_optuna']:
        if key in results[city]:
            m = results[city][key]
            ranking_rows.append({
                'City': CITY_LABELS[city], 'Model': MODEL_LABELS[key],
                'RMSE': m['RMSE'], 'MAE': m['MAE'], 'MAPE': m['MAPE'], 'R2': m['R2'],
            })

    city_multi = multi_run_df[
        (multi_run_df['City'] == CITY_LABELS[city]) &
        (multi_run_df['Model'].str.contains('CNN|LSTM', regex=True)) &
        (~multi_run_df['Model'].str.contains('Hybrid|XGBoost'))
    ]
    for _, row in city_multi.iterrows():
        ranking_rows.append({
            'City': CITY_LABELS[city], 'Model': f"{row['Model']} (5-run avg)",
            'RMSE': row['RMSE_mean'], 'MAE': row['MAE_mean'],
            'MAPE': row['MAPE_mean'], 'R2': row['R2_mean'],
        })

ranking_df = pd.DataFrame(ranking_rows)

for city in CITIES:
    city_name = CITY_LABELS[city]
    city_df = ranking_df[ranking_df['City'] == city_name].copy()

    city_df['RMSE_rank'] = city_df['RMSE'].rank(ascending=True)
    city_df['MAE_rank']  = city_df['MAE'].rank(ascending=True)
    city_df['MAPE_rank'] = city_df['MAPE'].rank(ascending=True)
    city_df['R2_rank']   = city_df['R2'].rank(ascending=False)

    city_df['Composite_Rank'] = city_df[['RMSE_rank','MAE_rank','MAPE_rank','R2_rank']].mean(axis=1)
    city_df = city_df.sort_values('Composite_Rank')

    print(f"\n{'='*70}\nCOMPOSITE RANKING — {city_name}\n{'='*70}")
    print(city_df[['Model', 'RMSE', 'MAE', 'MAPE', 'R2', 'Composite_Rank']].to_string(index=False))

ranking_df.to_csv(os.path.join(RESULTS_DIR, 'final_composite_ranking.csv'), index=False)
print(f"\n✅ Saved to: final_composite_ranking.csv")


COMPOSITE RANKING — Kano
                                 Model      RMSE       MAE      MAPE       R2  Composite_Rank
                    Standalone XGBoost 29.807397 17.084024 11.301194 0.984059            1.75
              Hybrid v3 (Optuna-Tuned) 30.250330 16.885340  9.142713 0.983582            2.00
      Hybrid v3 (Interactions + Tuned) 30.194244 17.308306 10.304926 0.983643            2.25
         Hybrid v2 (Enhanced Features) 30.524379 17.915525 11.658244 0.983283            4.50
                       Original Hybrid 30.547381 17.982955 11.588841 0.983258            5.00
           Standalone LSTM (5-run avg) 32.750679 21.235770 11.518332 0.980678            5.50
CNN-LSTM (uni-directional) (5-run avg) 44.010821 31.990519 28.401245 0.964852            7.00
            Standalone CNN (5-run avg) 53.561844 40.127150 28.918296 0.946980            8.00

COMPOSITE RANKING — Maiduguri
                                 Model      RMSE       MAE      MAPE       R2  Composite_Rank
   

In [11]:
# ============================================================
# OPTUNA TUNING - CELL 11: Tune Standalone XGBoost (fair comparison)
# ============================================================

def objective_standalone(trial, city):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 200, 600),
        'max_depth'         : trial.suggest_int('max_depth', 3, 8),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample'         : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight'  : trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 0.001, 1.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 0.1, 3.0, log=True),
        'objective'         : 'reg:squarederror',
        'random_state'      : 42,
        'n_jobs'            : -1,
        'verbosity'         : 0
    }

    X_train_flat = data[city]['X_train'][:, -1, :]
    X_val_flat   = data[city]['X_val'][:, -1, :]

    model = xgb.XGBRegressor(**params)
    model.fit(X_train_flat, data[city]['y_train'],
              eval_set=[(X_val_flat, data[city]['y_val'])], verbose=False)

    y_pred_val = model.predict(X_val_flat)
    y_true_kw = data[city]['tgt_scaler'].inverse_transform(data[city]['y_val'].reshape(-1,1)).flatten()
    y_pred_kw = data[city]['tgt_scaler'].inverse_transform(y_pred_val.reshape(-1,1)).flatten()
    mask = y_true_kw > 0
    return np.sqrt(mean_squared_error(y_true_kw[mask], y_pred_kw[mask]))


xgb_only_optuna_models = {}

for city in CITIES:
    print(f"\n{'='*55}\nOPTUNA SEARCH — Standalone XGBoost — {CITY_LABELS[city]}\n{'='*55}")

    study = optuna.create_study(direction='minimize')
    study.optimize(lambda trial: objective_standalone(trial, city), n_trials=N_TRIALS, show_progress_bar=True)

    print(f"Best validation RMSE: {study.best_value:.3f}")

    final_params = {**study.best_params, 'objective': 'reg:squarederror',
                     'random_state': 42, 'n_jobs': -1, 'verbosity': 0}

    X_train_flat = data[city]['X_train'][:, -1, :]
    X_val_flat   = data[city]['X_val'][:, -1, :]

    final_model = xgb.XGBRegressor(**final_params)
    final_model.fit(X_train_flat, data[city]['y_train'],
                     eval_set=[(X_val_flat, data[city]['y_val'])], verbose=False)

    xgb_only_optuna_models[city] = final_model
    joblib.dump(final_model, os.path.join(MODELS_DIR, f'xgb_only_{city}_optuna.pkl'))

    # Evaluate on test set
    X_test_flat = data[city]['X_test'][:, -1, :]
    y_pred_test = final_model.predict(X_test_flat)
    results[city]['xgb_only_optuna'] = evaluate_model(y_pred_test, data[city]['y_test'], data[city]['tgt_scaler'])

    m = results[city]['xgb_only_optuna']
    print(f"Test set — RMSE={m['RMSE']:.2f} | MAE={m['MAE']:.2f} | MAPE={m['MAPE']:.2f} | R2={m['R2']:.4f}")


OPTUNA SEARCH — Standalone XGBoost — Kano


  0%|          | 0/60 [00:00<?, ?it/s]

Best validation RMSE: 31.785
Test set — RMSE=29.38 | MAE=15.95 | MAPE=7.60 | R2=0.9845

OPTUNA SEARCH — Standalone XGBoost — Maiduguri


  0%|          | 0/60 [00:00<?, ?it/s]

Best validation RMSE: 30.226
Test set — RMSE=32.80 | MAE=18.06 | MAPE=11.40 | R2=0.9804

OPTUNA SEARCH — Standalone XGBoost — Abuja


  0%|          | 0/60 [00:00<?, ?it/s]

Best validation RMSE: 32.532
Test set — RMSE=30.79 | MAE=18.19 | MAPE=10.55 | R2=0.9812
